# Import libraries & data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

In [ ]:
path = 'drive/MyDrive/ViJudge/data/position_bias/statistics/'
output_path = 'drive/MyDrive/ViJudge/data/visualization/position_bias/'

In [ ]:
bias_df = pd.read_csv(path + 'bias_result.csv')
bias_df.head()

## Avg bias ratio

In [ ]:
avg_bias_df = bias_df.drop(columns=['attempt'])

# Calculate mean and standard deviation
mean_df = bias_df.groupby(['judge'])[['consistency', 'biased_toward_first', 'biased_toward_second', 'error']].mean().reset_index()
std_df = bias_df.groupby(['judge'])[['consistency', 'biased_toward_first', 'biased_toward_second', 'error']].std().fillna(0).reset_index()

avg_bias_df = mean_df.copy()

# Rename columns for clarity
mean_df = mean_df.rename(columns={'consistency': 'Consistency_mean', 'biased_toward_first': 'Biased toward first_mean', 'biased_toward_second': 'Biased toward second_mean', 'error': 'Error_mean'})
std_df = std_df.rename(columns={'consistency': 'Consistency_std', 'biased_toward_first': 'Biased toward first_std', 'biased_toward_second': 'Biased toward second_std', 'error': 'Error_std'})

# Merge mean and std dataframes
avg_bias_df = pd.merge(mean_df, std_df, on='judge')

# Apply string formatting to the columns with mean (±std) format
def format_percent_with_std(mean_val, std_val):
    return f'{mean_val*100:.2f} (\u00b1{std_val*100:.2f})'

avg_bias_df['Consistency (%)'] = avg_bias_df.apply(lambda row: format_percent_with_std(row['Consistency_mean'], row['Consistency_std']), axis=1)
avg_bias_df['Biased toward first (%)'] = avg_bias_df.apply(lambda row: format_percent_with_std(row['Biased toward first_mean'], row['Biased toward first_std']), axis=1)
avg_bias_df['Biased toward second (%)'] = avg_bias_df.apply(lambda row: format_percent_with_std(row['Biased toward second_mean'], row['Biased toward second_std']), axis=1)
avg_bias_df['Error (%)'] = avg_bias_df.apply(lambda row: format_percent_with_std(row['Error_mean'], row['Error_std']), axis=1)

# Select only the formatted columns and 'judge'
avg_bias_df = avg_bias_df[['judge', 'Consistency (%)', 'Biased toward first (%)', 'Biased toward second (%)', 'Error (%)']]

avg_bias_df.to_csv(output_path + 'position_avg_bias_ratio.csv', index=False)
display(avg_bias_df)